<a href="https://colab.research.google.com/github/vs-tries-to-code/flyrank-ml-starter/blob/week3/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (The Grain**)
One row = One unique URL/Page (page_id or url) at a specific snapshot point in time (e.g., snapshot as of 2026-03-31).

**Time Window**

**Feature Window**: Daily performance aggregated over the 60 days prior to the snapshot date (e.g., 2026-01-01 to 2026-02-28).

**Label Window**: Daily performance aggregated over the last 30 days before the snapshot date (e.g., 2026-04-01 to 2026-04-30).

**Tables to be used**
```dim_content``` and ```fact_content_query_90d```

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label**

```is_declining``` : A binary value which indicates whether the page is declining or not

</br>

**Features**
1. ```impressions_first30``` : Derived by (```impressions_90d``` - ```impressions_prev30``` -

```impressions_last30```) to prevent implicit data leakage

2. ```content_visible_query_count``` : The total number of unique search queries driving visibility to the page.

3. ```impressions_prev30``` : Direct performance from Days 31–60

4. ```days_since_last_update```  : Derived by ```content_updated_date``` - ```window_start```

5. ```avg_position_prev30d``` : Ranking baseline before target period

All the mentioned data is knowable at the time of decision making as it is already collected or derived from past data.

</br>

**Context**

1. Entity Identifiers: ```pseudonymized_client_id``` (or client hash), ```content_id``` / url, ```query_hash```

2. Categorical / Business Attributes: ```main_intent```, ```content_type```, ```competition_level```

3. Provenance / Creation Attributes: ```provider_used```, ```model_used```

</br>

**Excluded**

1. ```impressions_90d``` , ```clicks_90d```, ```avg_position_90d``` :  To prevent implicit data leakage while analysing.

2. ```is_deleted```  : Deletions could have been due to decline, so feeding this in as a feature would tamper with the prediction. All pages which were deleted would be directly flagged as declining.

3. ```last_optimised_date``` : If an optimization happened inside the label window, it distorts the target outcome.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
import duckdb
from google.colab import userdata

# 1. Retrieve the Hugging Face token you saved in Colab Secrets
hf_token = userdata.get('warehouse')

# 2. Connect to DuckDB
con = duckdb.connect()

# 3. Create the Hugging Face secret in DuckDB dynamically
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



**Counting distinct grain keys**

In [12]:
q1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(
        COALESCE(client_hash_id, ''), '_',
        COALESCE(content_hash_id, ''), '_',
        COALESCE(query_hash_id, '')
    )) AS distinct_grain_keys
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet');
"""


result = con.sql(q1)
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┐
│ total_rows │ distinct_grain_keys │
│   int64    │        int64        │
├────────────┼─────────────────────┤
│    2414248 │             2414248 │
└────────────┴─────────────────────┘



**Counting rows in along with earliest starting window and latest ending window to select specific dates for analysis**

In [20]:
q2 = f"""SELECT
    COUNT() AS slice_row_count,
    MIN(window_start) AS earliest_window_start,
    MAX(window_end) AS latest_window_end
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet'); """

result = con.sql(q2)
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬───────────────────────┬───────────────────┐
│ slice_row_count │ earliest_window_start │ latest_window_end │
│      int64      │         date          │       date        │
├─────────────────┼───────────────────────┼───────────────────┤
│         2414248 │ 2026-04-02            │ 2026-06-30        │
└─────────────────┴───────────────────────┴───────────────────┘



**The number rows which survive after removing deleted pages**

In [21]:
q3 = f"""SELECT
    COUNT(*) AS total_candidates,
    COUNT(CASE WHEN d.is_published IS TRUE AND d.is_deleted IS NOT TRUE THEN 1 END) AS surviving_published_rows,
    ROUND(
        100.0 * COUNT(CASE WHEN d.is_published IS TRUE AND d.is_deleted IS NOT TRUE THEN 1 END) / COUNT(*),
        2
    ) AS survival_percentage
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet') f
LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON f.content_hash_id = d.content_hash_id;"""

result = con.sql(q3)
print(result)

┌──────────────────┬──────────────────────────┬─────────────────────┐
│ total_candidates │ surviving_published_rows │ survival_percentage │
│      int64       │          int64           │       double        │
├──────────────────┼──────────────────────────┼─────────────────────┤
│          2414248 │                  2413941 │               99.99 │
└──────────────────┴──────────────────────────┴─────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. The data does not account for algorithm Updates (SERP Volatility), which may or may not lead to the decline of a page.

2. Internal metrics like search_volume and competition_level exists, but no live competitor data. It cannot be determined if a competitor published a far superior piece of content or launched a massive backlink campaign that outranked a page.

3. Word countand character count are proxy metrics, they do not equal quality.

4. The dataset lacks post-click behavior like bounce rate, dwell time, conversion rate, or scroll depth, and these factors cannot be predicted.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.